# PKG storage clean-up (cdi_tm quota)

The `cdi_tm` database folder has a 9.9 TB HDFS quota, counted as file size × replication. With 3 copies per file it
holds only about 3.3 TB of data; with 2 copies, about 4.9 TB.

1. **Settings.** The folders and tables to remove, the target replication, and `DRY_RUN`.
2. **Plan.** Shows what would be removed and how many of your files would go from 3 copies to 2, with the space
   each step frees. Nothing changes.
3. **Delete.** Removes the listed folders (skipping the trash, so the quota frees at once) and drops the listed tables.
4. **Replication.** Sets the replication of every file you own under the database folder to `TARGET_REPLICATION`.
   HDFS removes the extra copies in the background; the quota drops at once. Rerun this cell after future builds:
   new files are written with the cluster default (3) unless the writing notebook sets `dfs.replication`
   (the periodicity notebook now does).

Run with `DRY_RUN = True` first and check §2, then set `DRY_RUN = False` and run all. Do not run it while a periodicity
or customer-metrics build is going (their work and state folders are on the list). The TM build can keep running.

**Replication 2 vs 1.** With 2 copies, data survives one failed disk or node. With 1, any failed disk loses that file;
the tables can be rebuilt, but some take days. 2 is the recommended floor; the cluster may also refuse 1.

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
DRY_RUN = True                 # True = plan only
TARGET_REPLICATION = 2         # copies per file for every file you own under the database folder (3 = cluster default)

DELETE_PATHS = [               # relative to the database folder
    "_work_pkg_cust_reg",                          # periodicity work of the failed month; rebuilt at the next start
    "_state_pkg_cust_reg/month=2025-07",           # not needed: the build resumes at 2025-09 from the 2025-08 state
    "_state_pkg_cust_reg/month=2025-09",           # left by the failed attempt; rewritten when 2025-09 is built
    "_tmp_pkg_cpty_explore",                       # October exploration scratch
    "_pkg_cpty_metrics_state/313742575a",          # counterparty metrics state of two older signatures
    "_pkg_cpty_metrics_state/b11811d8b2",          #   (current: c6405f6803, used by the 2026-10 build: kept)
    "_tmp_pkg_cpty_metrics/313742575a",            # counterparty metrics work of the same older signatures
    "_tmp_pkg_cpty_metrics/b11811d8b2",
    "_tmp_pkg_cust_metrics_v3",                    # v3 work folder: v3 is no longer built
    "pkg_cust_churn_state",                        # v2 churn state: only needed to build another v2 month
    "pkg_cust_churn_state_v3",                     # v3 churn state: only needed to build another v3 month
    "pkg_cust_churn_state_v4/kind=reg",            # v4.0 periodicity stream state: v4.1 never reads it (its migration deletes it too)
    "_tmp_pkg_regularity",                         # pkg_payment_regularity / forecast-check scratch (analyses done)
    "_tmp_pkg_stop_spike",                         # pkg_stop_spike_check scratch (analysis done)
    "_tmp_pkg_cpty_map_v2",                        # entity map v2 build scratch (map built)
    "_tmp_pkg_fi_canonical",                       # canonical bank build scratch (tables built)
]
DROP_TABLES = [                # table definition dropped and its folder deleted
    # "pkg_cust_pair_month",        # v2 pair table, 0.25 TB: does fraud or the Streamlit app read it?
    # "pkg_cust_pair_month_v3",     # v3 pair table, 0.26 TB: does the attrition counterparty features notebook read it?
    # "pkg_cust_fi_month",          # v3 customer × bank (v4 has pkg_cust_fi_month_v4)
]
SKIP_RECENT_MIN = 30           # replication: files changed in the last N minutes are left alone (a running build may be writing them)

# ---- Setup ---------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)
import time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("pkg_storage_cleanup")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.executor.memoryOverhead", "1g").config("spark.driver.memory", "4g")
         .config("spark.dynamicAllocation.maxExecutors", "4")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass

HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
ME = spark._jvm.org.apache.hadoop.security.UserGroupInformation.getCurrentUser().getShortUserName()
ROOT_URI = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
FS = JPath(ROOT_URI).getFileSystem(HCONF)
def tb(v):
    return v / 1e12
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))
def table(pdf):
    return pdf.to_html(index=False, na_rep="", float_format=lambda x: f"{x:,.3f}")
def quota():
    c = FS.getContentSummary(JPath(ROOT_URI))
    return c.getSpaceConsumed(), c.getSpaceQuota(), c.getLength()
def jp(rel):
    return JPath(ROOT_URI + "/" + rel.strip("/"))
for p in DELETE_PATHS:
    if not p or p.startswith("/") or ".." in p or p.strip("/") == "":
        raise RuntimeError(f"DELETE_PATHS must be folders inside the database folder: {p!r}")
used0, q0, log0 = quota()
out(f"<p>Database folder <code>{_html.escape(ROOT_URI)}</code> · you are <b>{ME}</b> · "
    f"quota {tb(q0):,.2f} TB, used {tb(used0):,.2f} TB, free {tb(q0 - used0):,.3f} TB · DRY_RUN = {DRY_RUN}</p>")

In [ ]:
# ---- §2 Plan (changes nothing) -------------------------------------------------------------------
del_rows = []
for t in DROP_TABLES:
    loc = [x[1] for x in spark.sql(f"DESCRIBE TABLE EXTENDED {DB}.{t}").collect() if x[0] == "Location"]
    root_p, loc_p = JPath(ROOT_URI).toUri().getPath().rstrip("/"), (JPath(loc[0]).toUri().getPath().rstrip("/") if loc else "")
    if not loc_p.startswith(root_p + "/"):
        raise RuntimeError(f"{t} does not live inside the database folder ({loc_p}); not dropped")
    rel = loc_p[len(root_p):].strip("/")
    del_rows.append({"kind": "table", "name": t, "path": rel})
for p in DELETE_PATHS:
    del_rows.append({"kind": "folder", "name": "", "path": p.strip("/")})
for r in del_rows:
    ok = bool(r["path"]) and FS.exists(jp(r["path"]))
    c = FS.getContentSummary(jp(r["path"])) if ok else None
    r.update({"exists": "yes" if ok else "no", "TB used": tb(c.getSpaceConsumed()) if c else 0.0,
              "GB data": c.getLength() / 1e9 if c else 0.0})
DEL = pd.DataFrame(del_rows)
gone = set(r["path"] for r in del_rows if r["exists"] == "yes")

# replication of your remaining files (deleted folders left out)
t_cut = (time.time() - SKIP_RECENT_MIN * 60) * 1000
by_rep, todo_bytes, todo_files, recent = {}, 0, 0, 0
it = FS.listFiles(JPath(ROOT_URI), True)
root_len = len(JPath(ROOT_URI).toUri().getPath().rstrip("/"))
while it.hasNext():
    f = it.next()
    rel = f.getPath().toUri().getPath()[root_len:].strip("/")
    if any(rel == g or rel.startswith(g + "/") for g in gone) or f.getOwner() != ME:
        continue
    r_ = f.getReplication()
    k = by_rep.setdefault(r_, [0, 0]); k[0] += 1; k[1] += f.getLen()
    if r_ > TARGET_REPLICATION:
        if f.getModificationTime() > t_cut:
            recent += 1
        else:
            todo_files += 1; todo_bytes += f.getLen() * (r_ - TARGET_REPLICATION)
REP = pd.DataFrame([{"copies now": k, "files": v[0], "GB data": v[1] / 1e9, "TB used": tb(v[1] * k)} for k, v in sorted(by_rep.items())])
free_del = DEL["TB used"].sum() if len(DEL) else 0.0
SUM = pd.DataFrame([
    {"step": "now: used / quota (TB)", "TB": f"{tb(used0):,.2f} / {tb(q0):,.2f}"},
    {"step": "§3 delete frees (TB)", "TB": f"{free_del:,.2f}"},
    {"step": f"§4 replication → {TARGET_REPLICATION} frees (TB), {todo_files:,} files", "TB": f"{tb(todo_bytes):,.2f}"},
    {"step": "after both: used (TB)", "TB": f"{tb(used0) - free_del - tb(todo_bytes):,.2f}"},
    {"step": f"files changed in the last {SKIP_RECENT_MIN} min, left at their replication", "TB": f"{recent:,}"}])
out("<h3>§2 Plan</h3>", table(SUM), "<h4>To delete</h4>", table(DEL),
    f"<h4>Your remaining files by number of copies (owner {ME})</h4>", table(REP))

In [ ]:
# ---- §3 Delete ------------------------------------------------------------------------------------
res = []
if DRY_RUN:
    out("<h3>§3 Delete</h3><p>DRY_RUN = True: nothing deleted.</p>")
else:
    for r in del_rows:
        st = "missing"
        if r["kind"] == "table":
            spark.sql(f"DROP TABLE IF EXISTS {DB}.{r['name']}")
            st = "table dropped"
        if r["exists"] == "yes":
            FS.delete(jp(r["path"]), True)            # Hadoop API delete: no trash, quota frees at once
            st = (st + ", folder deleted") if r["kind"] == "table" else "deleted"
        res.append({"kind": r["kind"], "name": r["name"], "path": r["path"], "TB freed": r["TB used"], "status": st})
    u, q, _ = quota()
    out("<h3>§3 Delete</h3>", table(pd.DataFrame(res)),
        f"<p>Used now {tb(u):,.2f} TB of {tb(q):,.2f} TB (free {tb(q - u):,.2f} TB).</p>"
        + ("<p>Dropped tables: INVALIDATE METADATA in Impala.</p>" if DROP_TABLES else ""))

In [ ]:
# ---- §4 Replication → TARGET_REPLICATION for your files (progress updates in place) -----------------
if DRY_RUN:
    out(f"<h3>§4 Replication</h3><p>DRY_RUN = True: nothing changed ({todo_files:,} files would go to {TARGET_REPLICATION} copies).</p>")
else:
    ph = display(HTML("<p>§4 starting …</p>"), display_id=True)
    t0, done, failed, skipped, freed, fails = time.time(), 0, 0, 0, 0, []
    t_cut = (time.time() - SKIP_RECENT_MIN * 60) * 1000
    it = FS.listFiles(JPath(ROOT_URI), True)
    while it.hasNext():
        f = it.next()
        if f.getOwner() != ME or f.getReplication() <= TARGET_REPLICATION:
            continue
        if f.getModificationTime() > t_cut:
            skipped += 1
            continue
        try:
            if FS.setReplication(f.getPath(), TARGET_REPLICATION):
                done += 1; freed += f.getLen() * (f.getReplication() - TARGET_REPLICATION)
            else:
                failed += 1; fails.append(f.getPath().toString())
        except Exception as e:
            failed += 1; fails.append(f.getPath().toString() + " · " + str(e).splitlines()[0][:120])
        if (done + failed) % 2000 == 0:
            ph.update(HTML(f"<p>§4 {done:,} files set to {TARGET_REPLICATION} copies · {tb(freed):,.2f} TB freed · "
                           f"{failed:,} failed · {(time.time() - t0) / 60:.1f} min</p>"))
    u, q, _ = quota()
    ph.update(HTML("<div><h3>§4 Replication</h3>" + table(pd.DataFrame([
        {"item": f"files set to {TARGET_REPLICATION} copies", "value": f"{done:,}"},
        {"item": "space freed (TB)", "value": f"{tb(freed):,.2f}"},
        {"item": "failed (first 10 below)", "value": f"{failed:,}"},
        {"item": f"skipped: changed in the last {SKIP_RECENT_MIN} min", "value": f"{skipped:,}"},
        {"item": "used now / quota (TB)", "value": f"{tb(u):,.2f} / {tb(q):,.2f}"},
        {"item": "minutes", "value": f"{(time.time() - t0) / 60:.1f}"}]))
        + ("<p>" + "<br>".join(_html.escape(x) for x in fails[:10]) + "</p>" if fails else "")
        + "<p>HDFS removes the extra copies in the background; the quota already counts the new replication.</p></div>"))